# 02c Global LSTM Strict Forecasting

Train one pooled/global LSTM across all available ticker series with calendar-global purging and profit/risk validation selection.

In [1]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
for candidate in [cwd, cwd / "forecasting", cwd.parent, cwd.parent / "forecasting"]:
    if (candidate / "src" / "stock_forecast").exists():
        PROJECT_DIR = candidate
        break
else:
    raise RuntimeError("Cannot locate forecasting project directory with src/stock_forecast")

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ARTIFACT_DIR = PROJECT_DIR / "artifacts"
DATA_DIR = ARTIFACT_DIR / "data"
REPORTS_DIR = ARTIFACT_DIR / "reports"
GLOBAL_LSTM_DATA_DIR = DATA_DIR / "global_lstm"
GLOBAL_LSTM_OUTPUT_DIR = Path(os.environ.get("GLOBAL_LSTM_OUTPUT_DIR", ARTIFACT_DIR / "horizons")).expanduser().resolve()
for path in [DATA_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)


def display_path(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_DIR))
    except ValueError:
        return str(path)


print(f"PROJECT_DIR = {PROJECT_DIR}")

PROJECT_DIR = /home/sapce/forecasting_stock_prices/forecasting


In [2]:
import importlib.util
import pandas as pd
from IPython.display import display

from stock_forecast.artifacts import load_json, load_table
from stock_forecast.models import build_model
from stock_forecast.strict_protocol import run_strict_global_lstm_protocol

pd.set_option("display.max_columns", 200)

## Constants

In [3]:
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is required for this notebook. Install/use the torchlab environment.")

def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, default))


def env_bool(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y"}


def env_int_list(name: str, default: list[int]) -> list[int]:
    value = os.environ.get(name)
    if not value:
        return default
    return [int(item.strip()) for item in value.split(",") if item.strip()]


FORCE_RETRAIN = env_bool("GLOBAL_LSTM_FORCE_RETRAIN", True)
RANDOM_STATE = env_int("GLOBAL_LSTM_RANDOM_STATE", 42)

STRICT_VALIDATION_ROWS = env_int("STRICT_VALIDATION_ROWS", 126)
STRICT_TEST_ROWS = env_int("STRICT_TEST_ROWS", 126)
MATURE_MIN_ROWS = env_int("MATURE_MIN_ROWS", 1008)
LIMITED_HISTORY_MIN_BLOCK_ROWS = env_int("LIMITED_HISTORY_MIN_BLOCK_ROWS", 42)
MIN_TRAIN_ROWS = env_int("MIN_TRAIN_ROWS", 60)
STRICT_MAX_TRAIN_ROWS = env_int("STRICT_MAX_TRAIN_ROWS", 1260)
INNER_MAX_FOLDS = env_int("INNER_MAX_FOLDS", 3)
INNER_MIN_TRAIN_ROWS = env_int("INNER_MIN_TRAIN_ROWS", 504)

GLOBAL_LSTM_N_TRIALS = env_int("GLOBAL_LSTM_N_TRIALS", 30)
GLOBAL_LSTM_OPTUNA_N_JOBS = env_int("GLOBAL_LSTM_OPTUNA_N_JOBS", 1)
GLOBAL_LSTM_MAX_EPOCHS = env_int("GLOBAL_LSTM_MAX_EPOCHS", 120)
GLOBAL_LSTM_PATIENCE = env_int("GLOBAL_LSTM_PATIENCE", 12)
GLOBAL_LSTM_DEVICE = os.environ.get("GLOBAL_LSTM_DEVICE", "auto")
GLOBAL_LSTM_ENSEMBLE_SEEDS = env_int_list("GLOBAL_LSTM_ENSEMBLE_SEEDS", [1, 7, 21])

TRANSACTION_COST_BPS = 10
SLIPPAGE_BPS = 5
SIGNAL_ANCHOR = "expanding_median"
MIN_VALIDATION_TRADES = 8
MAX_VALIDATION_DRAWDOWN = -0.35

HORIZONS = [
    {"name": "week", "horizon": 5, "threshold_grid": [0.0, 0.0025, 0.005, 0.01]},
    {"name": "month", "horizon": 21, "threshold_grid": [0.0, 0.005, 0.01, 0.02]},
]
selected_horizons = {name.strip() for name in os.environ.get("GLOBAL_LSTM_HORIZONS", "").split(",") if name.strip()}
if selected_horizons:
    HORIZONS = [item for item in HORIZONS if item["name"] in selected_horizons]

## Model Config

In [4]:
def make_global_lstm_search_space(horizon: int) -> dict:
    huber_beta_choices = [0.04, 0.08, 0.12] if horizon >= 21 else [0.02, 0.04, 0.06]
    return {
        "lookback": {"type": "categorical", "choices": [40, 60, 90, 126]},
        "hidden_size": {"type": "categorical", "choices": [32, 64, 96, 128]},
        "num_layers": {"type": "categorical", "choices": [1, 2]},
        "input_projection_size": {"type": "categorical", "choices": [0, 64, 128]},
        "lstm_dropout": {"type": "float", "low": 0.0, "high": 0.35},
        "head_dropout": {"type": "float", "low": 0.10, "high": 0.50},
        "learning_rate": {"type": "float", "low": 1e-4, "high": 2e-3, "log": True},
        "weight_decay": {"type": "float", "low": 1e-5, "high": 1e-2, "log": True},
        "batch_size": {"type": "categorical", "choices": [64, 128, 256]},
        "loss": {"type": "categorical", "choices": ["smooth_l1", "mse"]},
        "huber_beta": {"type": "categorical", "choices": huber_beta_choices},
        "feature_clip": {"type": "categorical", "choices": [3.0, 5.0, 8.0]},
        "target_normalization": {"type": "categorical", "choices": ["global", "per_ticker"]},
        "balanced_ticker_sampling": {"type": "categorical", "choices": [True]},
        "grad_clip_norm": {"type": "float", "low": 0.5, "high": 2.0},
    }


def make_global_lstm_config(name: str, feature_cols: list[str], horizon: int) -> dict:
    return {
        "name": name,
        "model_type": "lstm",
        "estimator_factory": build_model,
        "input_mode": "full_frame",
        "feature_cols": feature_cols,
        "static_params": {
            "max_epochs": GLOBAL_LSTM_MAX_EPOCHS,
            "patience": GLOBAL_LSTM_PATIENCE,
            "device": GLOBAL_LSTM_DEVICE,
            "validation_fraction": 0.2,
        },
        "search_space": make_global_lstm_search_space(horizon),
        "post_selection_static_params": {"ensemble_seeds": GLOBAL_LSTM_ENSEMBLE_SEEDS},
        "n_trials": GLOBAL_LSTM_N_TRIALS,
        "optuna_n_jobs": GLOBAL_LSTM_OPTUNA_N_JOBS,
        "needs_scaler": False,
    }

## Load Global LSTM Data

In [5]:
horizon_inputs = []

for spec in HORIZONS:
    horizon_name = spec["name"]
    horizon = int(spec["horizon"])
    global_dir = GLOBAL_LSTM_DATA_DIR / "horizons" / horizon_name
    model_path = global_dir / "model_dataset.parquet"
    feature_path = global_dir / "feature_columns.json"
    if not model_path.exists() and not model_path.with_suffix(".csv").exists():
        raise FileNotFoundError(f"Missing global LSTM data for {horizon_name}. Run notebooks/01c_global_lstm_eda.ipynb first.")
    payload = load_json(feature_path)
    model_df = load_table(model_path)
    model_df["date"] = pd.to_datetime(model_df["date"])
    feature_sets = payload["global_feature_sets"]
    model_configs = [
        make_global_lstm_config("global_lstm_all", feature_sets["global_all"], horizon),
        make_global_lstm_config("global_lstm_stationary", feature_sets["global_stationary"], horizon),
    ]
    horizon_inputs.append({
        "horizon_name": horizon_name,
        "horizon": horizon,
        "threshold_grid": spec["threshold_grid"],
        "model_df": model_df,
        "feature_cols": feature_sets[payload.get("primary_global_feature_set", "global_stationary")],
        "target_col": payload["target_column"],
        "model_configs": model_configs,
        "artifact_dir": GLOBAL_LSTM_OUTPUT_DIR / horizon_name / "global_lstm",
    })
    print({
        "horizon": horizon_name,
        "rows": len(model_df),
        "tickers": model_df["ticker"].nunique(),
        "global_all_features": len(feature_sets["global_all"]),
        "global_stationary_features": len(feature_sets["global_stationary"]),
        "target": payload["target_column"],
    })

{'horizon': 'week', 'rows': 17880, 'tickers': 7, 'global_all_features': 167, 'global_stationary_features': 164, 'target': 'target_return_5_next_open'}
{'horizon': 'month', 'rows': 17768, 'tickers': 7, 'global_all_features': 167, 'global_stationary_features': 164, 'target': 'target_return_21_next_open'}


## Train And Evaluate

In [6]:
global_lstm_results = {}

for item in horizon_inputs:
    horizon_name = item["horizon_name"]
    horizon = int(item["horizon"])
    result = run_strict_global_lstm_protocol(
        model_df=item["model_df"],
        feature_cols=item["feature_cols"],
        target_col=item["target_col"],
        model_configs=item["model_configs"],
        artifact_dir=item["artifact_dir"],
        force_retrain=FORCE_RETRAIN,
        random_state=RANDOM_STATE,
        run_metadata={
            "horizon_name": horizon_name,
            "horizon": horizon,
            "training_notebook": "02c_global_lstm_forecasting",
            "global_lstm_run": True,
        },
        validation_rows=STRICT_VALIDATION_ROWS,
        test_rows=STRICT_TEST_ROWS,
        mature_min_rows=MATURE_MIN_ROWS,
        limited_history_min_block_rows=LIMITED_HISTORY_MIN_BLOCK_ROWS,
        min_train_rows=MIN_TRAIN_ROWS,
        max_train_rows=STRICT_MAX_TRAIN_ROWS,
        inner_max_folds=INNER_MAX_FOLDS,
        inner_min_train_rows=INNER_MIN_TRAIN_ROWS,
        transaction_cost_bps=TRANSACTION_COST_BPS,
        slippage_bps=SLIPPAGE_BPS,
        threshold_grid=item["threshold_grid"],
        signal_anchor=SIGNAL_ANCHOR,
        min_validation_trades=MIN_VALIDATION_TRADES,
        max_validation_drawdown=MAX_VALIDATION_DRAWDOWN,
    )
    global_lstm_results[horizon_name] = result

    print(f"=== Global LSTM strict protocol: {horizon_name} ({horizon} trading days) ===")
    display(result["validation_threshold_search"])
    display(result["validation_model_ranking"])
    display(result["test_panel_signal_metrics"])
    display(result["test_signal_metrics"])
    display(result["leakage_audit"])

    failed = result["leakage_audit"][~result["leakage_audit"]["passed"]]
    if not failed.empty:
        raise AssertionError(f"Global LSTM leakage audit failed for {horizon_name}: {failed['check'].tolist()}")

=== Global LSTM strict protocol: week (5 trading days) ===


,profit_risk_utility,panel_sharpe,mean_ticker_sharpe,cumulative_return,max_drawdown,turnover,number_of_trades,validation_constraints_pass,long_threshold,model_name,selected_threshold,horizon_name,horizon,training_notebook,global_lstm_run
0,1.085637,0.524420,1.712914,0.016225,-0.059477,0.014070,63,True,0.0100,global_lstm_all,0.01,week,5,02c_global_lstm_forecasting,True
1,0.877193,1.232192,0.924916,0.091850,-0.131288,0.040947,153,True,0.0000,global_lstm_all,0.01,week,5,02c_global_lstm_forecasting,True
2,0.797063,0.770205,1.068188,0.037824,-0.095665,0.028702,118,True,0.0050,global_lstm_all,0.01,week,5,02c_global_lstm_forecasting,True
3,0.796331,0.953840,0.957818,0.054692,-0.110890,0.033965,136,True,0.0025,global_lstm_all,0.01,week,5,02c_global_lstm_forecasting,True
4,1.643511,1.891479,1.818870,0.081763,-0.066449,0.039719,140,True,0.0100,global_lstm_stationary,0.01,week,5,02c_global_lstm_forecasting,True
5,1.609773,2.342594,1.454321,0.158668,-0.105853,0.039825,155,True,0.0050,global_lstm_stationary,0.01,week,5,02c_global_lstm_forecasting,True
6,1.325097,2.028084,1.175351,0.146044,-0.125537,0.040175,157,True,0.0025,global_lstm_stationary,0.01,week,5,02c_global_lstm_forecasting,True
7,1.272960,1.966566,1.125224,0.151792,-0.134225,0.037754,140,True,0.0000,global_lstm_stationary,0.01,week,5,02c_global_lstm_forecasting,True


,profit_risk_utility,panel_sharpe,mean_ticker_sharpe,cumulative_return,max_drawdown,turnover,number_of_trades,validation_constraints_pass,long_threshold,model_name,selected_threshold,horizon_name,horizon,training_notebook,global_lstm_run,validation_rank,is_validation_selected
4,1.643511,1.891479,1.818870,0.081763,-0.066449,0.039719,140,True,0.0100,global_lstm_stationary,0.01,week,5,02c_global_lstm_forecasting,True,1,True
5,1.609773,2.342594,1.454321,0.158668,-0.105853,0.039825,155,True,0.0050,global_lstm_stationary,0.01,week,5,02c_global_lstm_forecasting,True,2,False
6,1.325097,2.028084,1.175351,0.146044,-0.125537,0.040175,157,True,0.0025,global_lstm_stationary,0.01,week,5,02c_global_lstm_forecasting,True,3,False
7,1.272960,1.966566,1.125224,0.151792,-0.134225,0.037754,140,True,0.0000,global_lstm_stationary,0.01,week,5,02c_global_lstm_forecasting,True,4,False
0,1.085637,0.524420,1.712914,0.016225,-0.059477,0.014070,63,True,0.0100,global_lstm_all,0.01,week,5,02c_global_lstm_forecasting,True,5,False
1,0.877193,1.232192,0.924916,0.091850,-0.131288,0.040947,153,True,0.0000,global_lstm_all,0.01,week,5,02c_global_lstm_forecasting,True,6,False
2,0.797063,0.770205,1.068188,0.037824,-0.095665,0.028702,118,True,0.0050,global_lstm_all,0.01,week,5,02c_global_lstm_forecasting,True,7,False
3,0.796331,0.953840,0.957818,0.054692,-0.110890,0.033965,136,True,0.0025,global_lstm_all,0.01,week,5,02c_global_lstm_forecasting,True,8,False


,cumulative_return,annualized_return,annualized_volatility,periods_per_year,sharpe,sortino,max_drawdown,calmar,turnover,number_of_trades,ticker,model_name,signal_mode,n_rebalances,sample_warning,long_threshold
0,-0.049463,-0.095046,0.041808,252.0,-2.273410,-2.275084,-0.082686,-1.149482,0.026443,105,__panel__,global_lstm_all,overlapping_tranches,777,False,0.01
1,-0.063804,-0.121729,0.043652,252.0,-2.788582,-2.674919,-0.082013,-1.484265,0.029085,113,__panel__,global_lstm_stationary,overlapping_tranches,777,False,0.01


,cumulative_return,annualized_return,annualized_volatility,periods_per_year,sharpe,sortino,max_drawdown,calmar,turnover,number_of_trades,ticker,model_name,signal_mode,n_rebalances,sample_warning,long_threshold
0,-0.000340,-0.000681,0.061897,252.0,-0.010999,-0.009377,-0.065868,-0.010336,0.047619,30,CBOM,global_lstm_all,overlapping_tranches,126,False,0.01
1,-0.148774,-0.469664,0.066050,252.0,-7.110780,-7.315248,-0.153177,-3.066154,0.012500,4,MBNK,global_lstm_all,overlapping_tranches,64,False,0.01
2,-0.027403,-0.054056,0.025018,252.0,-2.160626,-1.275460,-0.036150,-1.495308,0.015873,10,SBER,global_lstm_all,overlapping_tranches,126,False,0.01
3,-0.035398,-0.069543,0.028052,252.0,-2.479117,-1.533042,-0.043339,-1.604627,0.009524,6,SBERP,global_lstm_all,overlapping_tranches,126,False,0.01
4,-0.082763,-0.230712,0.105979,252.0,-2.176966,-2.849456,-0.167397,-1.378237,0.038554,16,SVCB,global_lstm_all,overlapping_tranches,83,False,0.01
5,0.019625,0.039635,0.056859,252.0,0.697076,0.671712,-0.072459,0.547006,0.023810,15,T,global_lstm_all,overlapping_tranches,126,False,0.01
6,-0.077422,-0.148850,0.103657,252.0,-1.435983,-0.706076,-0.118305,-1.258185,0.038095,24,VTBR,global_lstm_all,overlapping_tranches,126,False,0.01
7,-0.005444,-0.010859,0.059452,252.0,-0.182652,-0.182768,-0.066393,-0.163556,0.041270,26,CBOM,global_lstm_stationary,overlapping_tranches,126,False,0.01
8,-0.104757,-0.353203,0.061276,252.0,-5.764106,-4.685464,-0.104757,-3.371651,0.031250,10,MBNK,global_lstm_stationary,overlapping_tranches,64,False,0.01
9,-0.004765,-0.009507,0.015063,252.0,-0.631145,-0.393403,-0.013567,-0.700713,0.022222,14,SBER,global_lstm_stationary,overlapping_tranches,126,False,0.01


...

,profit_risk_utility,panel_sharpe,mean_ticker_sharpe,cumulative_return,max_drawdown,turnover,number_of_trades,validation_constraints_pass,long_threshold,model_name,selected_threshold,horizon_name,horizon,training_notebook,global_lstm_run
0,4.308488,6.459286,3.624021,0.162874,-0.034680,0.003600,67,True,0.010,global_lstm_all,0.01,month,21,02c_global_lstm_forecasting,True
1,4.289264,6.358430,3.658260,0.175761,-0.046719,0.004279,81,True,0.005,global_lstm_all,0.01,month,21,02c_global_lstm_forecasting,True
2,4.047687,5.833747,3.570830,0.187188,-0.067414,0.004459,81,True,0.000,global_lstm_all,0.01,month,21,02c_global_lstm_forecasting,True
3,3.700150,5.665739,3.042951,0.121348,-0.021775,0.003297,60,True,0.020,global_lstm_all,0.01,month,21,02c_global_lstm_forecasting,True
4,3.513759,4.465539,3.533230,0.116276,-0.066468,0.002970,44,True,0.010,global_lstm_stationary,0.01,month,21,02c_global_lstm_forecasting,True
5,3.203749,4.420064,2.991627,0.146347,-0.081070,0.004156,64,True,0.000,global_lstm_stationary,0.01,month,21,02c_global_lstm_forecasting,True
6,3.023649,4.100340,2.882929,0.119527,-0.074544,0.003747,57,True,0.005,global_lstm_stationary,0.01,month,21,02c_global_lstm_forecasting,True
7,2.298286,2.739066,2.466974,0.061375,-0.062846,0.003101,39,True,0.020,global_lstm_stationary,0.01,month,21,02c_global_lstm_forecasting,True


,profit_risk_utility,panel_sharpe,mean_ticker_sharpe,cumulative_return,max_drawdown,turnover,number_of_trades,validation_constraints_pass,long_threshold,model_name,selected_threshold,horizon_name,horizon,training_notebook,global_lstm_run,validation_rank,is_validation_selected
0,4.308488,6.459286,3.624021,0.162874,-0.034680,0.003600,67,True,0.010,global_lstm_all,0.01,month,21,02c_global_lstm_forecasting,True,1,True
1,4.289264,6.358430,3.658260,0.175761,-0.046719,0.004279,81,True,0.005,global_lstm_all,0.01,month,21,02c_global_lstm_forecasting,True,2,False
2,4.047687,5.833747,3.570830,0.187188,-0.067414,0.004459,81,True,0.000,global_lstm_all,0.01,month,21,02c_global_lstm_forecasting,True,3,False
3,3.700150,5.665739,3.042951,0.121348,-0.021775,0.003297,60,True,0.020,global_lstm_all,0.01,month,21,02c_global_lstm_forecasting,True,4,False
4,3.513759,4.465539,3.533230,0.116276,-0.066468,0.002970,44,True,0.010,global_lstm_stationary,0.01,month,21,02c_global_lstm_forecasting,True,5,False
5,3.203749,4.420064,2.991627,0.146347,-0.081070,0.004156,64,True,0.000,global_lstm_stationary,0.01,month,21,02c_global_lstm_forecasting,True,6,False
6,3.023649,4.100340,2.882929,0.119527,-0.074544,0.003747,57,True,0.005,global_lstm_stationary,0.01,month,21,02c_global_lstm_forecasting,True,7,False
7,2.298286,2.739066,2.466974,0.061375,-0.062846,0.003101,39,True,0.020,global_lstm_stationary,0.01,month,21,02c_global_lstm_forecasting,True,8,False


,cumulative_return,annualized_return,annualized_volatility,periods_per_year,sharpe,sortino,max_drawdown,calmar,turnover,number_of_trades,ticker,model_name,signal_mode,n_rebalances,sample_warning,long_threshold
0,-0.072079,-0.136948,0.024585,252.0,-5.570309,-5.608611,-0.077351,-1.770477,0.004222,63,__panel__,global_lstm_all,overlapping_tranches,770,False,0.01
1,-0.122962,-0.227644,0.031428,252.0,-7.243331,-7.911197,-0.143197,-1.589721,0.002425,39,__panel__,global_lstm_stationary,overlapping_tranches,770,False,0.01


,cumulative_return,annualized_return,annualized_volatility,periods_per_year,sharpe,sortino,max_drawdown,calmar,turnover,number_of_trades,ticker,model_name,signal_mode,n_rebalances,sample_warning,long_threshold
0,-0.091626,-0.174857,0.050006,252.0,-3.496740,-3.473507,-0.095173,-1.837263,0.005669,15,CBOM,global_lstm_all,overlapping_tranches,126,False,0.01
1,-0.073862,-0.275501,0.053330,252.0,-5.165977,-9.568916,-0.095688,-2.879149,0.002381,3,MBNK,global_lstm_all,overlapping_tranches,60,False,0.01
2,-0.045204,-0.088365,0.019532,252.0,-4.524082,-3.546772,-0.056668,-1.559345,0.004157,11,SBER,global_lstm_all,overlapping_tranches,126,False,0.01
3,-0.041229,-0.080758,0.019349,252.0,-4.173707,-3.319278,-0.053757,-1.502297,0.003401,9,SBERP,global_lstm_all,overlapping_tranches,126,False,0.01
4,-0.172896,-0.450062,0.054240,252.0,-8.297566,-11.179525,-0.178216,-2.525370,0.002976,5,SVCB,global_lstm_all,overlapping_tranches,80,False,0.01
5,0.020362,0.041139,0.021483,252.0,1.914918,1.336406,-0.030467,1.350285,0.003401,9,T,global_lstm_all,overlapping_tranches,126,False,0.01
6,-0.101202,-0.192162,0.036176,252.0,-5.311805,-3.978419,-0.099344,-1.934311,0.004157,11,VTBR,global_lstm_all,overlapping_tranches,126,False,0.01
7,-0.111744,-0.211002,0.030392,252.0,-6.942577,-7.404267,-0.118140,-1.786026,0.001512,4,CBOM,global_lstm_stationary,overlapping_tranches,126,False,0.01
8,-0.081945,-0.301689,0.064345,252.0,-4.688625,-8.669343,-0.136236,-2.214458,0.002381,3,MBNK,global_lstm_stationary,overlapping_tranches,60,False,0.01
9,-0.053627,-0.104378,0.022051,252.0,-4.733574,-4.797009,-0.076807,-1.358971,0.003779,10,SBER,global_lstm_stationary,overlapping_tranches,126,False,0.01


,check,passed,details
0,global outer splits are available,True,split_rows=7
1,global validation predictions are available,True,rows=1540
2,global test predictions are available,True,rows=1540
3,global cutoffs are chronological,True,"global_validation_start=2024-11-22 00:00:00, g..."
4,ticker split dates are chronological,True,bad_rows=0
5,validation predictions match ticker split dates,True,"out_of_window=0, wrong_role=0"
6,test predictions match ticker split dates,True,"out_of_window=0, wrong_role=0"
7,validation model target dates end before globa...,True,overlap_rows=0
8,final refit target dates end before global tes...,True,overlap_rows=0
9,global final model payloads exist for test pre...,True,missing_models=0


## Comparison Handles

In [7]:
for item in horizon_inputs:
    horizon_name = item["horizon_name"]
    global_reports = item["artifact_dir"] / "strict_protocol" / "reports"
    main_reports = ARTIFACT_DIR / "horizons" / horizon_name / "strict_protocol" / "reports"
    print({
        "horizon": horizon_name,
        "global_lstm_reports": display_path(global_reports),
        "main_strict_reports": display_path(main_reports),
        "global_panel_metrics": display_path(global_reports / "test_panel_signal_metrics.parquet"),
    })

{'horizon': 'week', 'global_lstm_reports': 'artifacts/horizons/week/global_lstm/strict_protocol/reports', 'main_strict_reports': 'artifacts/horizons/week/strict_protocol/reports', 'global_panel_metrics': 'artifacts/horizons/week/global_lstm/strict_protocol/reports/test_panel_signal_metrics.parquet'}
{'horizon': 'month', 'global_lstm_reports': 'artifacts/horizons/month/global_lstm/strict_protocol/reports', 'main_strict_reports': 'artifacts/horizons/month/strict_protocol/reports', 'global_panel_metrics': 'artifacts/horizons/month/global_lstm/strict_protocol/reports/test_panel_signal_metrics.parquet'}
